# Data Preprocessing

## Overview

The Home Credit Default Risk dataset contains applicant information along with multiple transactional datasets representing bureau records, previous loan applications, installment payments, POS cash balances, and credit card activities.

The objective of this notebook is to transform these raw datasets into a clean, integrated, and applicant-level dataset suitable for statistical analysis and machine learning.

Unlike the previous notebook, which focused on understanding the data, this notebook focuses on data engineering and preprocessing.

---

## Objectives

The preprocessing pipeline includes the following steps:

- Load all datasets.
- Remove duplicate records.
- Handle missing values.
- Aggregate transactional datasets.
- Merge all datasets using the applicant identifier (`SK_ID_CURR`).
- Validate the integrated dataset.
- Export the processed dataset for downstream analysis.

The processed dataset generated in this notebook will be used for Exploratory Data Analysis (EDA), hypothesis testing, Principal Component Analysis (PCA), and machine learning model development.

# Import Required Libraries

This section imports the libraries required for data manipulation, preprocessing, and dataset integration.

In [1]:
# ============================================================
# Import Libraries
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

RANDOM_STATE = 42

# Load Datasets

The Home Credit dataset consists of multiple relational tables linked using unique identifiers.

Each dataset is loaded independently and will later be aggregated and merged to create a single applicant-level dataset.

In [2]:
# ============================================================
# Define Dataset Directory
# ============================================================

DATA_DIR = Path("../data")

In [3]:
# ============================================================
# Load Datasets
# ============================================================

application_df = pd.read_csv(DATA_DIR / "HC_application_train.csv")

bureau_df = pd.read_csv(DATA_DIR / "HC_bureau.csv")

bureau_balance_df = pd.read_csv(DATA_DIR / "HC_bureau_balance.csv")

previous_application_df = pd.read_csv(DATA_DIR / "HC_previous_application.csv")

installments_df = pd.read_csv(DATA_DIR / "HC_installments_payments.csv")

credit_card_df = pd.read_csv(DATA_DIR / "HC_credit_card_balance.csv")

pos_cash_df = pd.read_csv(DATA_DIR / "HC_POS_CASH_balance.csv")

# Remove Duplicate Records

Duplicate records can introduce bias into statistical analysis and machine learning models.

Although duplicate observations are uncommon in the Home Credit dataset, each dataset will be inspected to ensure data integrity before further preprocessing.

If duplicate records are identified, they will be removed while preserving the first occurrence.

In [4]:
# ============================================================
# Duplicate Record Summary
# ============================================================

datasets = {
    "Application": application_df,
    "Bureau": bureau_df,
    "Bureau Balance": bureau_balance_df,
    "Previous Applications": previous_application_df,
    "Installments": installments_df,
    "Credit Card": credit_card_df,
    "POS Cash": pos_cash_df
}

duplicate_summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Duplicate Rows": [df.duplicated().sum() for df in datasets.values()]
})

duplicate_summary

,Dataset,Duplicate Rows
0,Application,0
1,Bureau,0
2,Bureau Balance,0
3,Previous Applications,0
4,Installments,0
5,Credit Card,0
6,POS Cash,0


In [5]:
# ============================================================
# Remove Duplicate Records
# ============================================================

for name, df in datasets.items():
    before = df.shape[0]

    df.drop_duplicates(inplace=True)

    after = df.shape[0]

    print(f"{name:<25} Removed {before - after} duplicate rows")

Application               Removed 0 duplicate rows
Bureau                    Removed 0 duplicate rows
Bureau Balance            Removed 0 duplicate rows
Previous Applications     Removed 0 duplicate rows
Installments              Removed 0 duplicate rows
Credit Card               Removed 0 duplicate rows
POS Cash                  Removed 0 duplicate rows


# Handle Missing Values

Missing values are common in financial datasets and must be handled carefully to avoid introducing bias into predictive models.

Rather than applying a single strategy to all variables, missing values will be treated according to the percentage of missing data and the importance of each feature.

The preprocessing strategy adopted is:

- Less than **5%** missing values: Impute using statistical measures.
- Between **5% and 30%**: Evaluate feature importance and apply appropriate imputation.
- Greater than **30%**: Assess whether the feature should be retained or removed.

The following sections identify the missing value percentage for every feature in each dataset before treatment decisions are made.

In [6]:
# ============================================================
# Missing Value Summary Function
# ============================================================

def missing_value_summary(df):
    """
    Returns missing value statistics for a dataframe.
    """

    summary = pd.DataFrame({
        "Missing Count": df.isnull().sum(),
        "Missing Percentage":
            (df.isnull().mean() * 100).round(2)
    })

    summary = summary[summary["Missing Count"] > 0]

    return summary.sort_values(
        by="Missing Percentage",
        ascending=False
    )

In [7]:
# Missing Values in Application Dataset

application_missing = missing_value_summary(application_df)
bureau_missing = missing_value_summary(bureau_df)
bureau_balance_missing = missing_value_summary(bureau_balance_df)
previous_application_missing = missing_value_summary(previous_application_df)
installments_missing = missing_value_summary(installments_df)
credit_card_missing = missing_value_summary(credit_card_df)
pos_cash_missing = missing_value_summary(pos_cash_df)

application_missing.head(20)


,Missing Count,Missing Percentage
COMMONAREA_MEDI,214865,69.87
COMMONAREA_MODE,214865,69.87
COMMONAREA_AVG,214865,69.87
NONLIVINGAPARTMENTS_MODE,213514,69.43
NONLIVINGAPARTMENTS_MEDI,213514,69.43
NONLIVINGAPARTMENTS_AVG,213514,69.43
FONDKAPREMONT_MODE,210295,68.39
LIVINGAPARTMENTS_AVG,210199,68.35
LIVINGAPARTMENTS_MEDI,210199,68.35
LIVINGAPARTMENTS_MODE,210199,68.35


In [8]:
bureau_missing.head(20)


,Missing Count,Missing Percentage
AMT_ANNUITY,1226791,71.47
AMT_CREDIT_MAX_OVERDUE,1124488,65.51
DAYS_ENDDATE_FACT,633653,36.92
AMT_CREDIT_SUM_LIMIT,591780,34.48
AMT_CREDIT_SUM_DEBT,257669,15.01
DAYS_CREDIT_ENDDATE,105553,6.15
AMT_CREDIT_SUM,13,0.00


In [9]:
bureau_balance_missing.head(20)


,Missing Count,Missing Percentage


In [10]:
previous_application_missing.head(20)


,Missing Count,Missing Percentage
RATE_INTEREST_PRIVILEGED,1664263,99.64
RATE_INTEREST_PRIMARY,1664263,99.64
AMT_DOWN_PAYMENT,895844,53.64
RATE_DOWN_PAYMENT,895844,53.64
NAME_TYPE_SUITE,820405,49.12
DAYS_LAST_DUE,673065,40.30
DAYS_FIRST_DRAWING,673065,40.30
DAYS_FIRST_DUE,673065,40.30
DAYS_TERMINATION,673065,40.30
NFLAG_INSURED_ON_APPROVAL,673065,40.30


In [11]:
installments_missing.head(20)


,Missing Count,Missing Percentage
DAYS_ENTRY_PAYMENT,2905,0.02
AMT_PAYMENT,2905,0.02


In [12]:
credit_card_missing.head(20)


,Missing Count,Missing Percentage
AMT_PAYMENT_CURRENT,767988,20.00
AMT_DRAWINGS_ATM_CURRENT,749816,19.52
AMT_DRAWINGS_OTHER_CURRENT,749816,19.52
AMT_DRAWINGS_POS_CURRENT,749816,19.52
CNT_DRAWINGS_ATM_CURRENT,749816,19.52
CNT_DRAWINGS_POS_CURRENT,749816,19.52
CNT_DRAWINGS_OTHER_CURRENT,749816,19.52
AMT_INST_MIN_REGULARITY,305236,7.95
CNT_INSTALMENT_MATURE_CUM,305236,7.95


In [13]:
pos_cash_missing.head(20)

,Missing Count,Missing Percentage
CNT_INSTALMENT,26071,0.26
CNT_INSTALMENT_FUTURE,26087,0.26


# Observations from Missing Value Analysis

The missing value analysis reveals distinct patterns across the Home Credit datasets. The extent of missing data varies significantly depending on the type of information stored in each table. Therefore, dataset-specific preprocessing strategies will be adopted instead of applying a uniform treatment across all datasets.

### Application Dataset

- Several residential property-related features (e.g., `COMMONAREA_*`, `LIVINGAPARTMENTS_*`, `FLOORSMIN_*`, and `YEARS_BUILD_*`) contain more than **65% missing values**.
- `OWN_CAR_AGE` also exhibits a high proportion of missing values (65.99%). However, these missing values are likely structural, as applicants without a registered car naturally have no recorded car age.
- Most remaining variables have considerably lower missing percentages and can be treated through appropriate imputation techniques.

### Bureau Dataset

- `AMT_ANNUITY` (71.47%) and `AMT_CREDIT_MAX_OVERDUE` (65.51%) contain substantial missing values.
- Credit amount variables such as `AMT_CREDIT_SUM_DEBT` have relatively moderate missingness (15.01%), making them suitable for imputation.
- The Bureau dataset mainly contains historical credit information, making most variables valuable despite missing observations.

### Bureau Balance Dataset

- No missing values were identified.
- This dataset requires no missing value treatment before aggregation.

### Previous Application Dataset

- `RATE_INTEREST_PRIMARY` and `RATE_INTEREST_PRIVILEGED` contain approximately **99.6% missing values**, indicating insufficient information for reliable analysis.
- Several loan-related variables exhibit moderate missing percentages (20–54%) and will be retained through suitable imputation.
- Features associated with loan timelines also contain approximately 40% missing values and require careful treatment.

### Installments Payments Dataset

- Missing values account for less than **0.1%** of the dataset.
- These variables can be safely imputed without significantly affecting the overall data distribution.

### Credit Card Balance Dataset

- Transaction-related variables contain approximately **20%** missing values.
- Since these variables provide valuable behavioral information, they will be retained and imputed before aggregation.

### POS Cash Balance Dataset

- Missing values account for less than **0.3%** of the dataset.
- These missing observations are minimal and can be handled using simple statistical imputation.

Overall, the missing value analysis indicates that preprocessing decisions should balance information preservation with data quality. Variables with extremely high missing percentages and limited business relevance may be removed, whereas financially important variables will be retained and appropriately imputed.

# Missing Value Treatment Strategy

Based on the observations above, the following preprocessing strategy will be adopted throughout this project.

| Dataset | Strategy |
|----------|----------|
| **Application** | Remove highly sparse property-related features where appropriate, retain important credit-related variables, and impute the remaining missing values. |
| **Bureau** | Retain historical credit variables and impute missing numerical values using the median after assessing business relevance. |
| **Bureau Balance** | No missing value treatment required. |
| **Previous Application** | Remove variables with extremely high missing percentages (>95%), while retaining financially important variables through imputation. |
| **Installments Payments** | Apply median imputation to the small number of missing numerical values. |
| **Credit Card Balance** | Retain all variables and impute missing numerical values before aggregation. |
| **POS Cash Balance** | Apply median imputation due to the negligible proportion of missing values. |

The preprocessing strategy prioritizes preserving variables that contribute meaningful information for credit risk prediction while eliminating features with excessive missingness and limited analytical value.

# Drop Features with Excessive Missing Values

Several features in the Application dataset contain extremely high proportions of missing values, particularly those describing detailed residential property characteristics.

These variables provide limited information for the majority of applicants and are unlikely to contribute significantly to the prediction of credit default. Therefore, features with excessive missingness and low business relevance will be removed.

However, important variables such as `OWN_CAR_AGE` and the `EXT_SOURCE` features will be retained despite their missing values because they contain valuable predictive information for credit risk assessment.

In [14]:
# ============================================================
# Missing Value Treatment
# ============================================================

# ------------------------------------------------------------
# 1. Application Dataset
# ------------------------------------------------------------

# Drop highly sparse property-related features
application_drop_columns = [
    "COMMONAREA_AVG", "COMMONAREA_MEDI", "COMMONAREA_MODE",
    "NONLIVINGAPARTMENTS_AVG", "NONLIVINGAPARTMENTS_MEDI", "NONLIVINGAPARTMENTS_MODE",
    "LIVINGAPARTMENTS_AVG", "LIVINGAPARTMENTS_MEDI", "LIVINGAPARTMENTS_MODE",
    "FLOORSMIN_AVG", "FLOORSMIN_MEDI", "FLOORSMIN_MODE",
    "YEARS_BUILD_AVG", "YEARS_BUILD_MEDI", "YEARS_BUILD_MODE",
    "FONDKAPREMONT_MODE"
]

application_df.drop(columns=application_drop_columns, inplace=True, errors="ignore")

# Impute numerical features with median
application_num_cols = application_df.select_dtypes(include=["number"]).columns
application_df[application_num_cols] = application_df[application_num_cols].fillna(
    application_df[application_num_cols].median()
)

# Impute categorical features with mode
application_cat_cols = application_df.select_dtypes(include=["object"]).columns

for col in application_cat_cols:
    application_df[col].fillna(application_df[col].mode()[0], inplace=True)

In [15]:
# ------------------------------------------------------------
# 2. Bureau Dataset
# ------------------------------------------------------------

# Retain all variables
# Impute numerical variables using median

bureau_num_cols = bureau_df.select_dtypes(include=["number"]).columns

bureau_df[bureau_num_cols] = bureau_df[bureau_num_cols].fillna(
    bureau_df[bureau_num_cols].median()
)


In [16]:
# ------------------------------------------------------------
# 3. Bureau Balance Dataset
# ------------------------------------------------------------

# No missing values
# No preprocessing required

In [17]:
# ------------------------------------------------------------
# 4. Previous Application Dataset
# ------------------------------------------------------------

# Drop variables with >95% missing values
previous_drop_columns = [
    "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED"
]

previous_application_df.drop(
    columns=previous_drop_columns,
    inplace=True,
    errors="ignore"
)

# Numerical median imputation
previous_num_cols = previous_application_df.select_dtypes(include=["number"]).columns

previous_application_df[previous_num_cols] = previous_application_df[previous_num_cols].fillna(
    previous_application_df[previous_num_cols].median()
)

# Categorical mode imputation
previous_cat_cols = previous_application_df.select_dtypes(include=["object"]).columns

for col in previous_cat_cols:
    previous_application_df[col].fillna(
        previous_application_df[col].mode()[0],
        inplace=True
    )

In [18]:
# ------------------------------------------------------------
# 5. Installments Payments Dataset
# ------------------------------------------------------------

installments_num_cols = installments_df.select_dtypes(include=["number"]).columns

installments_df[installments_num_cols] = installments_df[installments_num_cols].fillna(
    installments_df[installments_num_cols].median()
)

In [19]:
# ------------------------------------------------------------
# 6. Credit Card Balance Dataset
# ------------------------------------------------------------

credit_card_num_cols = credit_card_df.select_dtypes(include=["number"]).columns

credit_card_df[credit_card_num_cols] = credit_card_df[credit_card_num_cols].fillna(
    credit_card_df[credit_card_num_cols].median()
)

In [20]:
# ------------------------------------------------------------
# 7. POS Cash Balance Dataset
# ------------------------------------------------------------

pos_cash_num_cols = pos_cash_df.select_dtypes(include=["number"]).columns

pos_cash_df[pos_cash_num_cols] = pos_cash_df[pos_cash_num_cols].fillna(
    pos_cash_df[pos_cash_num_cols].median()
)

print("Missing value treatment completed successfully.")

Missing value treatment completed successfully.


# Verify Missing Value Treatment

After applying the preprocessing strategy, each dataset is re-evaluated to ensure that the missing value treatment has been successfully implemented.

This validation step confirms that:

- Selected features have been removed.
- Numerical variables have been appropriately imputed.
- Categorical variables have been imputed where applicable.
- The datasets are ready for feature aggregation and integration.

In [21]:
# ============================================================
# Verify Remaining Missing Values
# ============================================================

datasets = {
    "Application": application_df,
    "Bureau": bureau_df,
    "Bureau Balance": bureau_balance_df,
    "Previous Application": previous_application_df,
    "Installments": installments_df,
    "Credit Card": credit_card_df,
    "POS Cash": pos_cash_df
}

verification = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Remaining Missing Values": [
        df.isnull().sum().sum() for df in datasets.values()
    ]
})

verification

,Dataset,Remaining Missing Values
0,Application,554076
1,Bureau,0
2,Bureau Balance,0
3,Previous Application,820751
4,Installments,0
5,Credit Card,0
6,POS Cash,0


# Observations After Missing Value Treatment

The verification results indicate that missing values have been successfully handled for the majority of the datasets.

| Dataset | Status |
|----------|--------|
| Bureau | No missing values remain. |
| Bureau Balance | No missing values remain. |
| Installments Payments | No missing values remain. |
| Credit Card Balance | No missing values remain. |
| POS Cash Balance | No missing values remain. |

However, two datasets continue to contain missing values:

- **Application Dataset:** 554,076 missing values remain.
- **Previous Application Dataset:** 820,751 missing values remain.

These remaining missing values are expected because certain variables were intentionally retained due to their business relevance. Removing these variables would result in the loss of potentially valuable information for credit risk prediction.

The remaining missing values will be examined to determine whether additional preprocessing or feature engineering is required before dataset aggregation.

# Identify Remaining Missing Features

To determine whether further preprocessing is required, the remaining missing values in the Application and Previous Application datasets are examined.

This step helps identify whether the remaining missing values are concentrated in a small number of important variables or distributed across multiple features.

In [22]:
# ============================================================
# Remaining Missing Values - Application Dataset
# ============================================================

application_remaining_missing = (
    application_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

application_remaining_missing = application_remaining_missing[
    application_remaining_missing > 0
]

application_remaining_missing

WALLSMATERIAL_MODE     156341
HOUSETYPE_MODE         154297
EMERGENCYSTATE_MODE    145755
OCCUPATION_TYPE         96391
NAME_TYPE_SUITE          1292
dtype: int64

In [23]:
# ============================================================
# Remaining Missing Values - Previous Application Dataset
# ============================================================

previous_remaining_missing = (
    previous_application_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

previous_remaining_missing = previous_remaining_missing[
    previous_remaining_missing > 0
]

previous_remaining_missing

NAME_TYPE_SUITE        820405
PRODUCT_COMBINATION       346
dtype: int64

# Treatment of Remaining Missing Values

After the initial preprocessing, only a small number of features continue to contain missing values.

These features are retained because they provide useful information for credit risk assessment and their missing values are not necessarily indicative of poor data quality.

The remaining variables will be treated individually according to their characteristics:

### Application Dataset

| Feature | Treatment |
|----------|-----------|
| `WALLSMATERIAL_MODE` | Impute using the mode. |
| `HOUSETYPE_MODE` | Impute using the mode. |
| `EMERGENCYSTATE_MODE` | Impute using the mode. |
| `OCCUPATION_TYPE` | Impute using the mode. |
| `NAME_TYPE_SUITE` | Impute using the mode. |

### Previous Application Dataset

| Feature | Treatment |
|----------|-----------|
| `NAME_TYPE_SUITE` | Impute using the mode. |
| `PRODUCT_COMBINATION` | Impute using the mode. |

Mode imputation is appropriate for these categorical variables because it preserves the most frequently occurring category without altering the overall distribution of the data.

In [24]:
# ============================================================
# Final Categorical Imputation
# ============================================================

# -----------------------------
# Application Dataset
# -----------------------------

application_remaining = [
    "WALLSMATERIAL_MODE",
    "HOUSETYPE_MODE",
    "EMERGENCYSTATE_MODE",
    "OCCUPATION_TYPE",
    "NAME_TYPE_SUITE"
]

for col in application_remaining:
    if col in application_df.columns:
        mode_value = application_df[col].mode(dropna=True)

        if not mode_value.empty:
            application_df[col] = application_df[col].fillna(mode_value.iloc[0])

# -----------------------------
# Previous Application Dataset
# -----------------------------

previous_remaining = [
    "NAME_TYPE_SUITE",
    "PRODUCT_COMBINATION"
]

for col in previous_remaining:
    if col in previous_application_df.columns:
        mode_value = previous_application_df[col].mode(dropna=True)

        if not mode_value.empty:
            previous_application_df[col] = previous_application_df[col].fillna(mode_value.iloc[0])

print("Categorical imputation completed.")

Categorical imputation completed.


In [25]:
# ============================================================
# Final Missing Value Verification
# ============================================================

datasets = {
    "Application": application_df,
    "Bureau": bureau_df,
    "Bureau Balance": bureau_balance_df,
    "Previous Application": previous_application_df,
    "Installments": installments_df,
    "Credit Card": credit_card_df,
    "POS Cash": pos_cash_df
}

final_missing_summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Remaining Missing Values": [
        df.isnull().sum().sum()
        for df in datasets.values()
    ]
})

final_missing_summary

,Dataset,Remaining Missing Values
0,Application,0
1,Bureau,0
2,Bureau Balance,0
3,Previous Application,0
4,Installments,0
5,Credit Card,0
6,POS Cash,0


In [26]:
# ============================================================
# Check Remaining Missing Columns
# ============================================================

print("Application Dataset")
print(application_df.isnull().sum()[application_df.isnull().sum() > 0])

print("\nPrevious Application Dataset")
print(previous_application_df.isnull().sum()[previous_application_df.isnull().sum() > 0])

Application Dataset
Series([], dtype: int64)

Previous Application Dataset
Series([], dtype: int64)


# Feature Aggregation

The Home Credit dataset consists of one primary application dataset and several transactional datasets containing multiple records for each applicant.

To create a unified applicant-level dataset, each transactional table is aggregated using the applicant identifier (`SK_ID_CURR`).

The aggregation process summarizes historical financial behavior by computing statistical measures such as:

- Mean
- Maximum
- Minimum
- Sum
- Count

The resulting aggregated datasets will contain one record per applicant and can be merged with the Application dataset for subsequent analysis and model development.

# Define Aggregation Function

A reusable aggregation function is defined to reduce code duplication and ensure a consistent preprocessing workflow across all transactional datasets.

The function groups records by `SK_ID_CURR` and computes summary statistics for numerical variables.

In [27]:
# ============================================================
# Generic Aggregation Function
# ============================================================

def aggregate_dataset(df, prefix, group_key="SK_ID_CURR"):
    """
    Aggregate numerical features with dataset-specific prefixes.
    """

    # Exclude identifier columns from aggregation
    id_columns = [
        "SK_ID_CURR",
        "SK_ID_PREV",
        "SK_ID_BUREAU"
    ]

    numerical_columns = [
        col for col in df.select_dtypes(include=["number"]).columns
        if col not in id_columns
    ]

    aggregation_dict = {
        col: ["mean", "max", "min", "sum"]
        for col in numerical_columns
    }

    aggregated = (
        df.groupby(group_key)
          .agg(aggregation_dict)
    )

    aggregated.columns = [
        f"{prefix}_{col}_{stat}".upper()
        for col, stat in aggregated.columns
    ]

    aggregated.reset_index(inplace=True)

    return aggregated

# Aggregate Bureau Dataset

The Bureau dataset contains historical credit records obtained import statistics

from external credit bureaus.

Each applicant may have multiple bureau records. These records are aggregated into a single applicant-level representation by calculating summary statistics for all numerical variables.

In [28]:
# ============================================================
# Aggregate Bureau Dataset
# ============================================================

bureau_agg = aggregate_dataset(bureau_df, prefix="bureau")

print("Shape:", bureau_agg.shape)

bureau_agg.head()

Shape: (305811, 49)


,SK_ID_CURR,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_SUM,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_DAY_OVERDUE_MIN,BUREAU_CREDIT_DAY_OVERDUE_SUM,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_CREDIT_ENDDATE_MAX,BUREAU_DAYS_CREDIT_ENDDATE_MIN,BUREAU_DAYS_CREDIT_ENDDATE_SUM,BUREAU_DAYS_ENDDATE_FACT_MEAN,BUREAU_DAYS_ENDDATE_FACT_MAX,BUREAU_DAYS_ENDDATE_FACT_MIN,BUREAU_DAYS_ENDDATE_FACT_SUM,BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN,BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX,BUREAU_AMT_CREDIT_MAX_OVERDUE_MIN,BUREAU_AMT_CREDIT_MAX_OVERDUE_SUM,BUREAU_CNT_CREDIT_PROLONG_MEAN,BUREAU_CNT_CREDIT_PROLONG_MAX,BUREAU_CNT_CREDIT_PROLONG_MIN,BUREAU_CNT_CREDIT_PROLONG_SUM,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_MIN,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_MIN,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN,BUREAU_AMT_CREDIT_SUM_LIMIT_MAX,BUREAU_AMT_CREDIT_SUM_LIMIT_MIN,BUREAU_AMT_CREDIT_SUM_LIMIT_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX,BUREAU_AMT_CREDIT_SUM_OVERDUE_MIN,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MAX,BUREAU_DAYS_CREDIT_UPDATE_MIN,BUREAU_DAYS_CREDIT_UPDATE_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_AMT_ANNUITY_MAX,BUREAU_AMT_ANNUITY_MIN,BUREAU_AMT_ANNUITY_SUM
0,100001,-735.00,-49,-1572,-5145,0.00,0,0,0,82.43,1778.00,-1329.00,577.00,-856.14,-544.00,-1328.00,-5993.00,0.00,0.00,0.00,0.00,0.00,0,0,0,207623.57,378000.00,85500.00,1453365.00,85240.93,373239.00,0.00,596686.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-93.14,-6,-155,-652,3545.36,10822.50,0.00,24817.50
1,100002,-874.00,-103,-1437,-6992,0.00,0,0,0,-344.25,780.00,-1072.00,-2754.00,-747.38,-36.00,-1185.00,-5979.00,1050.64,5043.65,0.00,8405.15,0.00,0,0,0,108131.95,450000.00,0.00,865055.56,30722.62,245781.00,0.00,245781.00,3998.57,31988.56,0.00,31988.56,0.00,0.00,0.00,0.00,-499.88,-7,-1185,-3999,0.00,0.00,0.00,0.00
2,100003,-1400.75,-606,-2586,-5603,0.00,0,0,0,-544.50,1216.00,-2434.00,-2178.00,-1047.25,-540.00,-2131.00,-4189.00,0.00,0.00,0.00,0.00,0.00,0,0,0,254350.12,810000.00,22248.00,1017400.50,0.00,0.00,0.00,0.00,202500.00,810000.00,0.00,810000.00,0.00,0.00,0.00,0.00,-816.00,-43,-2131,-3264,0.00,0.00,0.00,0.00
3,100004,-867.00,-408,-1326,-1734,0.00,0,0,0,-488.50,-382.00,-595.00,-977.00,-532.50,-382.00,-683.00,-1065.00,0.00,0.00,0.00,0.00,0.00,0,0,0,94518.90,94537.80,94500.00,189037.80,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-532.00,-382,-682,-1064,0.00,0.00,0.00,0.00
4,100005,-190.67,-62,-373,-572,0.00,0,0,0,439.33,1324.00,-128.00,1318.00,-639.00,-123.00,-897.00,-1917.00,0.00,0.00,0.00,0.00,0.00,0,0,0,219042.00,568800.00,29826.00,657126.00,189469.50,543087.00,0.00,568408.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-54.33,-11,-121,-163,1420.50,4261.50,0.00,4261.50


# Aggregate Previous Application Dataset

The Previous Application dataset contains historical loan applications submitted by each applicant.

Aggregating these records captures an applicant's previous borrowing behavior and financial history.

In [29]:
# ============================================================
# Aggregate Previous Applications
# ============================================================

previous_agg = aggregate_dataset(previous_application_df, prefix="previous")

print("Shape:", previous_agg.shape)

previous_agg.head()

Shape: (338857, 69)


,SK_ID_CURR,PREVIOUS_AMT_ANNUITY_MEAN,PREVIOUS_AMT_ANNUITY_MAX,PREVIOUS_AMT_ANNUITY_MIN,PREVIOUS_AMT_ANNUITY_SUM,PREVIOUS_AMT_APPLICATION_MEAN,PREVIOUS_AMT_APPLICATION_MAX,PREVIOUS_AMT_APPLICATION_MIN,PREVIOUS_AMT_APPLICATION_SUM,PREVIOUS_AMT_CREDIT_MEAN,PREVIOUS_AMT_CREDIT_MAX,PREVIOUS_AMT_CREDIT_MIN,PREVIOUS_AMT_CREDIT_SUM,PREVIOUS_AMT_DOWN_PAYMENT_MEAN,PREVIOUS_AMT_DOWN_PAYMENT_MAX,PREVIOUS_AMT_DOWN_PAYMENT_MIN,PREVIOUS_AMT_DOWN_PAYMENT_SUM,PREVIOUS_AMT_GOODS_PRICE_MEAN,PREVIOUS_AMT_GOODS_PRICE_MAX,PREVIOUS_AMT_GOODS_PRICE_MIN,PREVIOUS_AMT_GOODS_PRICE_SUM,PREVIOUS_HOUR_APPR_PROCESS_START_MEAN,PREVIOUS_HOUR_APPR_PROCESS_START_MAX,PREVIOUS_HOUR_APPR_PROCESS_START_MIN,PREVIOUS_HOUR_APPR_PROCESS_START_SUM,PREVIOUS_NFLAG_LAST_APPL_IN_DAY_MEAN,PREVIOUS_NFLAG_LAST_APPL_IN_DAY_MAX,PREVIOUS_NFLAG_LAST_APPL_IN_DAY_MIN,PREVIOUS_NFLAG_LAST_APPL_IN_DAY_SUM,PREVIOUS_RATE_DOWN_PAYMENT_MEAN,PREVIOUS_RATE_DOWN_PAYMENT_MAX,PREVIOUS_RATE_DOWN_PAYMENT_MIN,PREVIOUS_RATE_DOWN_PAYMENT_SUM,PREVIOUS_DAYS_DECISION_MEAN,PREVIOUS_DAYS_DECISION_MAX,PREVIOUS_DAYS_DECISION_MIN,PREVIOUS_DAYS_DECISION_SUM,PREVIOUS_SELLERPLACE_AREA_MEAN,PREVIOUS_SELLERPLACE_AREA_MAX,PREVIOUS_SELLERPLACE_AREA_MIN,PREVIOUS_SELLERPLACE_AREA_SUM,PREVIOUS_CNT_PAYMENT_MEAN,PREVIOUS_CNT_PAYMENT_MAX,PREVIOUS_CNT_PAYMENT_MIN,PREVIOUS_CNT_PAYMENT_SUM,PREVIOUS_DAYS_FIRST_DRAWING_MEAN,PREVIOUS_DAYS_FIRST_DRAWING_MAX,PREVIOUS_DAYS_FIRST_DRAWING_MIN,PREVIOUS_DAYS_FIRST_DRAWING_SUM,PREVIOUS_DAYS_FIRST_DUE_MEAN,PREVIOUS_DAYS_FIRST_DUE_MAX,PREVIOUS_DAYS_FIRST_DUE_MIN,PREVIOUS_DAYS_FIRST_DUE_SUM,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_MEAN,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_MAX,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_MIN,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_SUM,PREVIOUS_DAYS_LAST_DUE_MEAN,PREVIOUS_DAYS_LAST_DUE_MAX,PREVIOUS_DAYS_LAST_DUE_MIN,PREVIOUS_DAYS_LAST_DUE_SUM,PREVIOUS_DAYS_TERMINATION_MEAN,PREVIOUS_DAYS_TERMINATION_MAX,PREVIOUS_DAYS_TERMINATION_MIN,PREVIOUS_DAYS_TERMINATION_SUM,PREVIOUS_NFLAG_INSURED_ON_APPROVAL_MEAN,PREVIOUS_NFLAG_INSURED_ON_APPROVAL_MAX,PREVIOUS_NFLAG_INSURED_ON_APPROVAL_MIN,PREVIOUS_NFLAG_INSURED_ON_APPROVAL_SUM
0,100001,3951.00,3951.00,3951.00,3951.00,24835.50,24835.50,24835.50,24835.50,23787.00,23787.00,23787.00,23787.00,2520.00,2520.00,2520.00,2520.00,24835.50,24835.50,24835.50,24835.50,13.00,13,13,13,1.00,1,1,1,0.10,0.10,0.10,0.10,-1740.00,-1740,-1740,-1740,23.00,23,23,23,8.00,8.00,8.00,8.00,365243.00,365243.00,365243.00,365243.00,-1709.00,-1709.00,-1709.00,-1709.00,-1499.00,-1499.00,-1499.00,-1499.00,-1619.00,-1619.00,-1619.00,-1619.00,-1612.00,-1612.00,-1612.00,-1612.00,0.00,0.00,0.00,0.00
1,100002,9251.77,9251.77,9251.77,9251.77,179055.00,179055.00,179055.00,179055.00,179055.00,179055.00,179055.00,179055.00,0.00,0.00,0.00,0.00,179055.00,179055.00,179055.00,179055.00,9.00,9,9,9,1.00,1,1,1,0.00,0.00,0.00,0.00,-606.00,-606,-606,-606,500.00,500,500,500,24.00,24.00,24.00,24.00,365243.00,365243.00,365243.00,365243.00,-565.00,-565.00,-565.00,-565.00,125.00,125.00,125.00,125.00,-25.00,-25.00,-25.00,-25.00,-17.00,-17.00,-17.00,-17.00,0.00,0.00,0.00,0.00
2,100003,56553.99,98356.99,6737.31,169661.97,435436.50,900000.00,68809.50,1306309.50,484191.00,1035882.00,68053.50,1452573.00,2841.00,6885.00,0.00,8523.00,435436.50,900000.00,68809.50,1306309.50,14.67,17,12,44,1.00,1,1,3,0.05,0.10,0.00,0.15,-1305.00,-746,-2341,-3915,533.00,1400,-1,1599,10.00,12.00,6.00,30.00,365243.00,365243.00,365243.00,1095729.00,-1274.33,-716.00,-2310.00,-3823.00,-1004.33,-386.00,-1980.00,-3013.00,-1054.33,-536.00,-1980.00,-3163.00,-1047.33,-527.00,-1976.00,-3142.00,0.67,1.00,0.00,2.00
3,100004,5357.25,5357.25,5357.25,5357.25,24282.00,24282.00,24282.00,24282.00,20106.00,20106.00,20106.00,20106.00,4860.00,4860.00,4860.00,4860.00,24282.00,24282.00,24282.00,24282.00,5.00,5,5,5,1.00,1,1,1,0.21,0.21,0.21,0.21,-815.00,-815,-815,-815,30.00,30,30,30,4.00,4.00,4.00,4.00,365243.00,365243.00,365243.00,365243.00,-784.00,-784.00,-784.00,-784.00,-694.00,-694.00,-694.00,-694.00,-724.00,-724.00,-724.00,-724.00,-714.00,-714.

# Aggregate Installments Payments Dataset

The Installments Payments dataset records historical repayment transactions for previous loans.

Aggregation summarizes repayment behavior for each applicant.

In [30]:
# ============================================================
# Aggregate Installments
# ============================================================

installments_agg = aggregate_dataset(installments_df, prefix="installments")

print("Shape:", installments_agg.shape)

installments_agg.head()

Shape: (339587, 25)


,SK_ID_CURR,INSTALLMENTS_NUM_INSTALMENT_VERSION_MEAN,INSTALLMENTS_NUM_INSTALMENT_VERSION_MAX,INSTALLMENTS_NUM_INSTALMENT_VERSION_MIN,INSTALLMENTS_NUM_INSTALMENT_VERSION_SUM,INSTALLMENTS_NUM_INSTALMENT_NUMBER_MEAN,INSTALLMENTS_NUM_INSTALMENT_NUMBER_MAX,INSTALLMENTS_NUM_INSTALMENT_NUMBER_MIN,INSTALLMENTS_NUM_INSTALMENT_NUMBER_SUM,INSTALLMENTS_DAYS_INSTALMENT_MEAN,INSTALLMENTS_DAYS_INSTALMENT_MAX,INSTALLMENTS_DAYS_INSTALMENT_MIN,INSTALLMENTS_DAYS_INSTALMENT_SUM,INSTALLMENTS_DAYS_ENTRY_PAYMENT_MEAN,INSTALLMENTS_DAYS_ENTRY_PAYMENT_MAX,INSTALLMENTS_DAYS_ENTRY_PAYMENT_MIN,INSTALLMENTS_DAYS_ENTRY_PAYMENT_SUM,INSTALLMENTS_AMT_INSTALMENT_MEAN,INSTALLMENTS_AMT_INSTALMENT_MAX,INSTALLMENTS_AMT_INSTALMENT_MIN,INSTALLMENTS_AMT_INSTALMENT_SUM,INSTALLMENTS_AMT_PAYMENT_MEAN,INSTALLMENTS_AMT_PAYMENT_MAX,INSTALLMENTS_AMT_PAYMENT_MIN,INSTALLMENTS_AMT_PAYMENT_SUM
0,100001,1.14,2.00,1.00,8.00,2.71,4,1,19,-2187.71,-1619.00,-2916.00,-15314.00,-2195.00,-1628.00,-2916.00,-15365.00,5885.13,17397.90,3951.00,41195.93,5885.13,17397.90,3951.00,41195.93
1,100002,1.05,2.00,1.00,20.00,10.00,19,1,190,-295.00,-25.00,-565.00,-5605.00,-315.42,-49.00,-587.00,-5993.00,11559.25,53093.75,9251.77,219625.70,11559.25,53093.75,9251.77,219625.70
2,100003,1.04,2.00,1.00,26.00,5.08,12,1,127,-1378.16,-536.00,-2310.00,-34454.00,-1385.32,-544.00,-2324.00,-34633.00,64754.59,560835.36,6662.97,1618864.65,64754.59,560835.36,6662.97,1618864.65
3,100004,1.33,2.00,1.00,4.00,2.00,3,1,6,-754.00,-724.00,-784.00,-2262.00,-761.67,-727.00,-795.00,-2285.00,7096.15,10573.97,5357.25,21288.47,7096.15,10573.97,5357.25,21288.47
4,100005,1.11,2.00,1.00,10.00,5.00,9,1,45,-586.00,-466.00,-706.00,-5274.00,-609.56,-470.00,-736.00,-5486.00,6240.20,17656.24,4813.20,56161.85,6240.20,17656.24,4813.20,56161.85


# Aggregate Credit Card Balance Dataset

The Credit Card Balance dataset contains monthly credit card utilization records.

Aggregating these records provides applicant-level credit card usage statistics.

In [31]:
# ============================================================
# Aggregate Credit Card Dataset
# ============================================================

credit_card_agg = aggregate_dataset(credit_card_df,prefix="credit_card")

print("Shape:", credit_card_agg.shape)

credit_card_agg.head()

Shape: (103558, 81)


,SK_ID_CURR,CREDIT_CARD_MONTHS_BALANCE_MEAN,CREDIT_CARD_MONTHS_BALANCE_MAX,CREDIT_CARD_MONTHS_BALANCE_MIN,CREDIT_CARD_MONTHS_BALANCE_SUM,CREDIT_CARD_AMT_BALANCE_MEAN,CREDIT_CARD_AMT_BALANCE_MAX,CREDIT_CARD_AMT_BALANCE_MIN,CREDIT_CARD_AMT_BALANCE_SUM,CREDIT_CARD_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CREDIT_CARD_AMT_CREDIT_LIMIT_ACTUAL_MAX,CREDIT_CARD_AMT_CREDIT_LIMIT_ACTUAL_MIN,CREDIT_CARD_AMT_CREDIT_LIMIT_ACTUAL_SUM,CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_MEAN,CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_MAX,CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_MIN,CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_SUM,CREDIT_CARD_AMT_DRAWINGS_CURRENT_MEAN,CREDIT_CARD_AMT_DRAWINGS_CURRENT_MAX,CREDIT_CARD_AMT_DRAWINGS_CURRENT_MIN,CREDIT_CARD_AMT_DRAWINGS_CURRENT_SUM,CREDIT_CARD_AMT_DRAWINGS_OTHER_CURRENT_MEAN,CREDIT_CARD_AMT_DRAWINGS_OTHER_CURRENT_MAX,CREDIT_CARD_AMT_DRAWINGS_OTHER_CURRENT_MIN,CREDIT_CARD_AMT_DRAWINGS_OTHER_CURRENT_SUM,CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_MEAN,CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_MAX,CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_MIN,CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_SUM,CREDIT_CARD_AMT_INST_MIN_REGULARITY_MEAN,CREDIT_CARD_AMT_INST_MIN_REGULARITY_MAX,CREDIT_CARD_AMT_INST_MIN_REGULARITY_MIN,CREDIT_CARD_AMT_INST_MIN_REGULARITY_SUM,CREDIT_CARD_AMT_PAYMENT_CURRENT_MEAN,CREDIT_CARD_AMT_PAYMENT_CURRENT_MAX,CREDIT_CARD_AMT_PAYMENT_CURRENT_MIN,CREDIT_CARD_AMT_PAYMENT_CURRENT_SUM,CREDIT_CARD_AMT_PAYMENT_TOTAL_CURRENT_MEAN,CREDIT_CARD_AMT_PAYMENT_TOTAL_CURRENT_MAX,CREDIT_CARD_AMT_PAYMENT_TOTAL_CURRENT_MIN,CREDIT_CARD_AMT_PAYMENT_TOTAL_CURRENT_SUM,CREDIT_CARD_AMT_RECEIVABLE_PRINCIPAL_MEAN,CREDIT_CARD_AMT_RECEIVABLE_PRINCIPAL_MAX,CREDIT_CARD_AMT_RECEIVABLE_PRINCIPAL_MIN,CREDIT_CARD_AMT_RECEIVABLE_PRINCIPAL_SUM,CREDIT_CARD_AMT_RECIVABLE_MEAN,CREDIT_CARD_AMT_RECIVABLE_MAX,CREDIT_CARD_AMT_RECIVABLE_MIN,CREDIT_CARD_AMT_RECIVABLE_SUM,CREDIT_CARD_AMT_TOTAL_RECEIVABLE_MEAN,CREDIT_CARD_AMT_TOTAL_RECEIVABLE_MAX,CREDIT_CARD_AMT_TOTAL_RECEIVABLE_MIN,CREDIT_CARD_AMT_TOTAL_RECEIVABLE_SUM,CREDIT_CARD_CNT_DRAWINGS_ATM_CURRENT_MEAN,CREDIT_CARD_CNT_DRAWINGS_ATM_CURRENT_MAX,CREDIT_CARD_CNT_DRAWINGS_ATM_CURRENT_MIN,CREDIT_CARD_CNT_DRAWINGS_ATM_CURRENT_SUM,CREDIT_CARD_CNT_DRAWINGS_CURRENT_MEAN,CREDIT_CARD_CNT_DRAWINGS_CURRENT_MAX,CREDIT_CARD_CNT_DRAWINGS_CURRENT_MIN,CREDIT_CARD_CNT_DRAWINGS_CURRENT_SUM,CREDIT_CARD_CNT_DRAWINGS_OTHER_CURRENT_MEAN,CREDIT_CARD_CNT_DRAWINGS_OTHER_CURRENT_MAX,CREDIT_CARD_CNT_DRAWINGS_OTHER_CURRENT_MIN,CREDIT_CARD_CNT_DRAWINGS_OTHER_CURRENT_SUM,CREDIT_CARD_CNT_DRAWINGS_POS_CURRENT_MEAN,CREDIT_CARD_CNT_DRAWINGS_POS_CURRENT_MAX,CREDIT_CARD_CNT_DRAWINGS_POS_CURRENT_MIN,CREDIT_CARD_CNT_DRAWINGS_POS_CURRENT_SUM,CREDIT_CARD_CNT_INSTALMENT_MATURE_CUM_MEAN,CREDIT_CARD_CNT_INSTALMENT_MATURE_CUM_MAX,CREDIT_CARD_CNT_INSTALMENT_MATURE_CUM_MIN,CREDIT_CARD_CNT_INSTALMENT_MATURE_CUM_SUM,CREDIT_CARD_SK_DPD_MEAN,CREDIT_CARD_SK_DPD_MAX,CREDIT_CARD_SK_DPD_MIN,CREDIT_CARD_SK_DPD_SUM,CREDIT_CARD_SK_DPD_DEF_MEAN,CREDIT_CARD_SK_DPD_DEF_MAX,CREDIT_CARD_SK_DPD_DEF_MIN,CREDIT_CARD_SK_DPD_DEF_SUM
0,100006,-3.50,-1,-6,-21,0.00,0.00,0.00,0.00,270000.00,270000,270000,1620000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2702.70,2702.70,2702.70,16216.20,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,0,0.00,0,0,0
1,100011,-38.50,-2,-75,-2849,54482.11,189000.00,0.00,4031676.23,164189.19,180000,90000,12150000,2432.43,180000.00,0.00,180000.00,2432.43,180000.00,0.00,180000.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,3902.76,9000.00,0.00,288804.20,4843.06,55485.00,0.00,358386.75,4520.07,55485.00,0.00,334485.00,52402.09,180000.00,0.00,3877754.58,54433.18,189000.00,-563.36,4028055.25,54433.18,189000.00,-563.36,4028055.25,0.05,4.00,0.00,4.00,0.05,4,0,4,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,25.62,33.00,1.00,1896.00,0.00,0,0,0,0.00,0,0,0
2,100013,-48.50,-1,-96,-4656,18159.92,161420.22,0.00,1743352.25,131718.75,1

# Aggregate POS Cash Balance Dataset

The POS Cash Balance dataset contains point-of-sale and cash loan repayment histories.

Aggregation summarizes repayment characteristics for each applicant.

In [32]:
# ============================================================
# Aggregate POS Cash Dataset
# ============================================================

pos_cash_agg = aggregate_dataset(pos_cash_df, prefix="pos_cash")

print("Shape:", pos_cash_agg.shape)

pos_cash_agg.head()

Shape: (337252, 21)


,SK_ID_CURR,POS_CASH_MONTHS_BALANCE_MEAN,POS_CASH_MONTHS_BALANCE_MAX,POS_CASH_MONTHS_BALANCE_MIN,POS_CASH_MONTHS_BALANCE_SUM,POS_CASH_CNT_INSTALMENT_MEAN,POS_CASH_CNT_INSTALMENT_MAX,POS_CASH_CNT_INSTALMENT_MIN,POS_CASH_CNT_INSTALMENT_SUM,POS_CASH_CNT_INSTALMENT_FUTURE_MEAN,POS_CASH_CNT_INSTALMENT_FUTURE_MAX,POS_CASH_CNT_INSTALMENT_FUTURE_MIN,POS_CASH_CNT_INSTALMENT_FUTURE_SUM,POS_CASH_SK_DPD_MEAN,POS_CASH_SK_DPD_MAX,POS_CASH_SK_DPD_MIN,POS_CASH_SK_DPD_SUM,POS_CASH_SK_DPD_DEF_MEAN,POS_CASH_SK_DPD_DEF_MAX,POS_CASH_SK_DPD_DEF_MIN,POS_CASH_SK_DPD_DEF_SUM
0,100001,-72.56,-53,-96,-653,4.00,4.00,4.00,36.00,1.44,4.00,0.00,13.00,0.78,7,0,7,0.78,7,0,7
1,100002,-10.00,-1,-19,-190,24.00,24.00,24.00,456.00,15.00,24.00,6.00,285.00,0.00,0,0,0,0.00,0,0,0
2,100003,-43.79,-18,-77,-1226,10.11,12.00,6.00,283.00,5.79,12.00,0.00,162.00,0.00,0,0,0,0.00,0,0,0
3,100004,-25.50,-24,-27,-102,3.75,4.00,3.00,15.00,2.25,4.00,0.00,9.00,0.00,0,0,0,0.00,0,0,0
4,100005,-20.00,-15,-25,-220,11.73,12.00,9.00,129.00,7.18,12.00,0.00,79.00,0.00,0,0,0,0.00,0,0,0


# Aggregate Bureau Balance Dataset

The Bureau Balance dataset contains monthly status records for each credit account identified by `SK_ID_BUREAU`.

Since this dataset does not contain the applicant identifier (`SK_ID_CURR`), it must first be aggregated at the credit account level and then merged with the Bureau dataset. The enriched Bureau dataset can subsequently be aggregated to the applicant level.

This two-stage aggregation preserves valuable historical repayment information while ensuring compatibility with the applicant-level dataset.

In [33]:
# ============================================================
# Aggregate Bureau Balance by SK_ID_BUREAU
# ============================================================

# Numerical features
bureau_balance_num = (
    bureau_balance_df
    .groupby("SK_ID_BUREAU")
    .agg({
        "MONTHS_BALANCE": ["min", "max", "mean", "count"]
    })
)

bureau_balance_num.columns = [
    f"BB_{col}_{stat}".upper()
    for col, stat in bureau_balance_num.columns
]

# Categorical feature (STATUS)
bureau_balance_status = (
    bureau_balance_df
    .groupby("SK_ID_BUREAU")["STATUS"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    .to_frame("BB_STATUS_MODE")
)

# Combine
bureau_balance_agg = bureau_balance_num.join(bureau_balance_status)

bureau_balance_agg.reset_index(inplace=True)

print("Shape:", bureau_balance_agg.shape)

bureau_balance_agg.head()

Shape: (817395, 6)


,SK_ID_BUREAU,BB_MONTHS_BALANCE_MIN,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_MEAN,BB_MONTHS_BALANCE_COUNT,BB_STATUS_MODE
0,5001709,-96,0,-48.00,97,C
1,5001710,-82,0,-41.00,83,C
2,5001711,-3,0,-1.50,4,0
3,5001712,-18,0,-9.00,19,0
4,5001713,-21,0,-10.50,22,X


# Merge Bureau and Bureau Balance

The aggregated Bureau Balance features are merged with the Bureau dataset using the common key `SK_ID_BUREAU`.

This enriches each credit record with historical monthly repayment information before aggregating the data at the applicant level.

In [34]:
# ============================================================
# Merge Bureau with Bureau Balance
# ============================================================

bureau_enriched = bureau_df.merge(
    bureau_balance_agg,
    on="SK_ID_BUREAU",
    how="left"
)

print("Shape:", bureau_enriched.shape)

bureau_enriched.head()

Shape: (1716428, 22)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,BB_MONTHS_BALANCE_MIN,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_MEAN,BB_MONTHS_BALANCE_COUNT,BB_STATUS_MODE
0,215354,5714462,Closed,currency 1,-497,0,-153.00,-153.00,0.00,0,91323.00,0.00,0.00,0.00,Consumer credit,-131,0.00,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.00,-897.00,0.00,0,225000.00,171342.00,0.00,0.00,Credit card,-20,0.00,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,-203,0,528.00,-897.00,0.00,0,464323.50,0.00,0.00,0.00,Consumer credit,-16,0.00,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,-203,0,-330.00,-897.00,0.00,0,90000.00,0.00,0.00,0.00,Credit card,-16,0.00,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.00,-897.00,77674.50,0,2700000.00,0.00,0.00,0.00,Consumer credit,-21,0.00,NaN,NaN,NaN,NaN,NaN


# Aggregate Enriched Bureau Dataset

After incorporating the Bureau Balance information, the enriched Bureau dataset is aggregated by `SK_ID_CURR`.

This produces a single applicant-level representation that combines external credit history with monthly account status information.

In [35]:
# ============================================================
# Aggregate Enriched Bureau Dataset
# ============================================================

bureau_final_agg = aggregate_dataset(
    bureau_enriched,
    prefix="BUREAU"
)

print("Shape:", bureau_final_agg.shape)

bureau_final_agg.head()

Shape: (305811, 65)


,SK_ID_CURR,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_SUM,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_DAY_OVERDUE_MIN,BUREAU_CREDIT_DAY_OVERDUE_SUM,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_CREDIT_ENDDATE_MAX,BUREAU_DAYS_CREDIT_ENDDATE_MIN,BUREAU_DAYS_CREDIT_ENDDATE_SUM,BUREAU_DAYS_ENDDATE_FACT_MEAN,BUREAU_DAYS_ENDDATE_FACT_MAX,BUREAU_DAYS_ENDDATE_FACT_MIN,BUREAU_DAYS_ENDDATE_FACT_SUM,BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN,BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX,BUREAU_AMT_CREDIT_MAX_OVERDUE_MIN,BUREAU_AMT_CREDIT_MAX_OVERDUE_SUM,BUREAU_CNT_CREDIT_PROLONG_MEAN,BUREAU_CNT_CREDIT_PROLONG_MAX,BUREAU_CNT_CREDIT_PROLONG_MIN,BUREAU_CNT_CREDIT_PROLONG_SUM,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_MIN,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_MIN,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN,BUREAU_AMT_CREDIT_SUM_LIMIT_MAX,BUREAU_AMT_CREDIT_SUM_LIMIT_MIN,BUREAU_AMT_CREDIT_SUM_LIMIT_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX,BUREAU_AMT_CREDIT_SUM_OVERDUE_MIN,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MAX,BUREAU_DAYS_CREDIT_UPDATE_MIN,BUREAU_DAYS_CREDIT_UPDATE_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_AMT_ANNUITY_MAX,BUREAU_AMT_ANNUITY_MIN,BUREAU_AMT_ANNUITY_SUM,BUREAU_BB_MONTHS_BALANCE_MIN_MEAN,BUREAU_BB_MONTHS_BALANCE_MIN_MAX,BUREAU_BB_MONTHS_BALANCE_MIN_MIN,BUREAU_BB_MONTHS_BALANCE_MIN_SUM,BUREAU_BB_MONTHS_BALANCE_MAX_MEAN,BUREAU_BB_MONTHS_BALANCE_MAX_MAX,BUREAU_BB_MONTHS_BALANCE_MAX_MIN,BUREAU_BB_MONTHS_BALANCE_MAX_SUM,BUREAU_BB_MONTHS_BALANCE_MEAN_MEAN,BUREAU_BB_MONTHS_BALANCE_MEAN_MAX,BUREAU_BB_MONTHS_BALANCE_MEAN_MIN,BUREAU_BB_MONTHS_BALANCE_MEAN_SUM,BUREAU_BB_MONTHS_BALANCE_COUNT_MEAN,BUREAU_BB_MONTHS_BALANCE_COUNT_MAX,BUREAU_BB_MONTHS_BALANCE_COUNT_MIN,BUREAU_BB_MONTHS_BALANCE_COUNT_SUM
0,100001,-735.00,-49,-1572,-5145,0.00,0,0,0,82.43,1778.00,-1329.00,577.00,-856.14,-544.00,-1328.00,-5993.00,0.00,0.00,0.00,0.00,0.00,0,0,0,207623.57,378000.00,85500.00,1453365.00,85240.93,373239.00,0.00,596686.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-93.14,-6,-155,-652,3545.36,10822.50,0.00,24817.50,-23.57,-1.00,-51.00,-165.00,0.00,0.00,0.00,0.00,-11.79,-0.50,-25.50,-82.50,24.57,52.00,2.00,172.00
1,100002,-874.00,-103,-1437,-6992,0.00,0,0,0,-344.25,780.00,-1072.00,-2754.00,-747.38,-36.00,-1185.00,-5979.00,1050.64,5043.65,0.00,8405.15,0.00,0,0,0,108131.95,450000.00,0.00,865055.56,30722.62,245781.00,0.00,245781.00,3998.57,31988.56,0.00,31988.56,0.00,0.00,0.00,0.00,-499.88,-7,-1185,-3999,0.00,0.00,0.00,0.00,-28.25,-3.00,-47.00,-226.00,-15.50,0.00,-32.00,-124.00,-21.88,-1.50,-39.50,-175.00,13.75,22.00,4.00,110.00
2,100003,-1400.75,-606,-2586,-5603,0.00,0,0,0,-544.50,1216.00,-2434.00,-2178.00,-1047.25,-540.00,-2131.00,-4189.00,0.00,0.00,0.00,0.00,0.00,0,0,0,254350.12,810000.00,22248.00,1017400.50,0.00,0.00,0.00,0.00,202500.00,810000.00,0.00,810000.00,0.00,0.00,0.00,0.00,-816.00,-43,-2131,-3264,0.00,0.00,0.00,0.00,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.00
3,100004,-867.00,-408,-1326,-1734,0.00,0,0,0,-488.50,-382.00,-595.00,-977.00,-532.50,-382.00,-683.00,-1065.00,0.00,0.00,0.00,0.00,0.00,0,0,0,94518.90,94537.80,94500.00,189037.80,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-532.00,-382,-682,-1064,0.00,0.00,0.00,0.00,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.00
4,100005,-190.67,-62,-373,-572,0.00,0,0,0,439.33,1324.00,-128.00,1318.00,-639.00,-123.00,-897.00,-1917.00,0.00,0.00,0.00,0.00,0.00,0,0,0,219042.00,568800.00,29826.00,657126.00,189469.50,543087.00,0.00,568408.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-54.33,-11,-121,-163,1420.50,4261.50,0.00,4261.50,-6.00,-2.00,-12.00,-18.00,0.00,0.00,0.00,0.00,-3.00,-1.00,-6.00,-9.00,7.00,13.00,3.00,21.00


In [36]:
previous_agg = aggregate_dataset(
    previous_application_df,
    prefix="PREV"
)

installments_agg = aggregate_dataset(
    installments_df,
    prefix="INST"
)

credit_card_agg = aggregate_dataset(
    credit_card_df,
    prefix="CC"
)

pos_cash_agg = aggregate_dataset(
    pos_cash_df,
    prefix="POS"
)

# Merge All Aggregated Datasets

The final preprocessing step combines the primary Application dataset with all aggregated transactional datasets.

A left join is performed using `SK_ID_CURR`, ensuring that every applicant in the Application dataset is retained even if corresponding transactional records are unavailable.

The resulting merged dataset forms the foundation for exploratory data analysis, hypothesis testing, feature engineering, dimensionality reduction, and machine learning model development.

In [37]:
# ============================================================
# Merge All Aggregated Datasets
# ============================================================

merged_df = application_df.copy()

# Merge aggregated datasets
merged_df = merged_df.merge(bureau_final_agg, on="SK_ID_CURR", how="left")
merged_df = merged_df.merge(previous_agg, on="SK_ID_CURR", how="left")
merged_df = merged_df.merge(installments_agg, on="SK_ID_CURR", how="left")
merged_df = merged_df.merge(credit_card_agg, on="SK_ID_CURR", how="left")
merged_df = merged_df.merge(pos_cash_agg, on="SK_ID_CURR", how="left")

print("Merged Dataset Shape:", merged_df.shape)

merged_df.head()

Merged Dataset Shape: (307511, 362)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,LANDAREA_AVG,LIVINGAREA_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,LANDAREA_MODE,LIVINGAREA_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,LANDAREA_MEDI,LIVINGAREA_MEDI,NONLIVINGAREA_MEDI,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_SUM,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_DAY_OVERDUE_MIN,BUREAU_CREDIT_DAY_OVERDUE_SUM,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_CREDIT_ENDDATE_MAX,BUREAU_DAYS_CREDIT_ENDDATE_MIN,BUREAU_DAYS_CREDIT_ENDDATE_SUM,BUREAU_DAYS_ENDDATE_FACT_MEAN,BUREAU_DAYS_ENDDATE_FACT_MAX,BUREAU_DAYS_ENDDATE_FACT_MIN,BUREAU_DAYS_ENDDATE_FACT_SUM,BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN,BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX,BUREAU_AMT_CREDIT_MAX_OVERDUE_MIN,BUREAU_AMT_CREDIT_MAX_OVERDUE_SUM,BUREAU_CNT_CREDIT_PROLONG_MEAN,BUREAU_CNT_CREDIT_PROLONG_MAX,BUREAU_CNT_CREDIT_PROLONG_MIN,BUREAU_CNT_CREDIT_PROLONG_SUM,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_MIN,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_MIN,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN,BUREAU_AMT_CREDIT_SUM_LIMIT_MAX,BUREAU_AMT_CREDIT_SUM_LIMIT_MIN,BUREAU_AMT_CREDIT_SUM_LIMIT_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX,BUREAU_AMT_CREDIT_SUM_OVERDUE_MIN,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MAX,BUREAU_DAYS_CREDIT_UPDATE_MIN,BUREAU_DAYS_CREDIT_UPDATE_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_AMT_ANNUITY_MAX,BUREAU_AMT_ANNUITY_MIN,BUREAU_AMT_ANNUITY_SUM,BUREAU_BB_MONTHS_BALANCE_MIN_MEAN,BUREAU_BB_MONTHS_BALANCE_MIN_MAX,BUREAU_BB_MONTHS_BALANCE_MIN_MIN,BUREAU_BB_MONTHS_BALANCE_MIN_SUM,BUREAU_BB_MONTHS_BALANCE_MAX_MEAN,BUREAU_BB_MONTHS_BALANCE_MAX_MAX,BUREAU_BB_MONTHS_BALANCE_MAX_MIN,BUREAU_BB_MONTHS_BALANCE_MAX_SUM,BUREAU_BB_MONTHS_BALANCE_MEAN_MEAN,BUREAU_BB_MONTHS_BALANCE_MEAN_MAX,BUREAU_BB_MONTHS_BALANCE_MEAN_MIN,BUREAU_BB_MONTHS_BALANCE_MEAN_SUM,BUREAU_BB_MONTHS_BALANCE_COUNT_MEAN,BUREAU_BB_MONTHS_BALANCE_COUNT_MAX,BUREAU_BB_MONTHS_BALANCE_COUNT_MIN,BUREAU_BB_MONTHS_BALANCE_COUNT_SUM,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,PREV_AMT_ANNUITY_MIN,PREV_AMT_ANNUITY_SUM,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATI

# Validate the Merged Dataset

After merging all datasets, it is important to verify the integrity of the final dataset.

The validation includes:

- Confirming the number of rows and columns.
- Checking for duplicate applicant IDs.
- Examining the overall missing value count.
- Ensuring that the dataset is ready for exploratory data analysis and feature engineering.

In [38]:
# ============================================================
# Final Dataset Validation
# ============================================================

print(f"Rows: {merged_df.shape[0]:,}")
print(f"Columns: {merged_df.shape[1]:,}")

print("\nDuplicate SK_ID_CURR:",
      merged_df["SK_ID_CURR"].duplicated().sum())

print("\nTotal Missing Values:",
      merged_df.isnull().sum().sum())

Rows: 307,511
Columns: 362

Duplicate SK_ID_CURR: 0

Total Missing Values: 24381924


# Save the Processed Dataset

The fully preprocessed and merged applicant-level dataset is saved as a CSV file.

Saving the dataset ensures that subsequent notebooks can be executed independently without repeating the preprocessing pipeline. This also improves reproducibility and reduces computation time.

In [39]:
# ============================================================
# Save Processed Dataset
# ============================================================

from pathlib import Path

# Create processed data directory if it does not exist
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save merged dataset
output_path = processed_dir / "credit_risk_processed.csv"

merged_df.to_csv(output_path, index=False)

print(f"Processed dataset saved to:\n{output_path}")
print(f"Dataset Shape: {merged_df.shape}")

Processed dataset saved to:
..\data\processed\credit_risk_processed.csv
Dataset Shape: (307511, 362)
